# Semana 01 · AgentOps y LLMOps con MLflow

Práctica guiada para observar trazas y evaluación mínima sin depender de un proveedor LLM. Usaremos un agente determinista para aprender el flujo: entrada → ruta → contexto → respuesta → traza → evaluación.


## 0. Dependencias y configuración

Deja `USE_LLM = False`. El objetivo es instrumentar y evaluar, no llamar a un modelo externo.


In [0]:
%pip install -q --upgrade "mlflow[databricks]"

In [0]:
import json
from typing import Any

import mlflow

USE_LLM = False
AGENT_VERSION = "v1-deterministic"
EXPERIMENT_NAME = "/Shared/muiaap-operacion-modelos/semana01-agentops"
mlflow.set_experiment(EXPERIMENT_NAME)
mlflow.set_tags({"course": "operacion-modelos", "week": "01", "agent.version": AGENT_VERSION})

{"ts": "2026-10-04 10:17:06.820", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMTcwODUxNjg0Nzg5NjM4NRABIAEyJDAxYTEwNjZhLThhODgtN2IxNy1hN2U1LTlkMmUxYWFiNzZmODokNzM2MGIxNjgtMzkxMi0zODdiLWEzNGEtMmM3ZTI2NWU3N2IxSgsIkM+I1gYQgKToA1ABWAFgAWi8/8TS9MSjDYgBAA==.", "context": {}}
{"ts": "2026-10-04 10:17:06.820", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMTcwODUxNjg0Nzg5NjM4NRABIAEyJDAxYTEwNjZhLThhODgtN2IxNy1hN2U1LTlkMmUxYWFiNzZmODokNzM2MGIxNjgtMzkxMi0zODdiLWEzNGEtMmM3ZTI2NWU3N2IxSgsIkM+I1gYQgKToA1ABWAFgAWi8/8TS9MSjDYgBAA==.", "context": {}}
{"ts": "2026-10-04 10:17:06.820", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMTcwODUxNjg0Nzg5NjM4NRABIAEyJDAxYTEwNjZhLThhODgtN2IxNy1hN2U1LTlkMmUxYWFiNzZmODokNzM2MGIxNjgtMzkxMi0zODdiLWEzNGEtMmM3ZTI2

## 1. Trazar un agente sencillo

**TODO 1:** completa las reglas de ruta, recuperación y respuesta. Decide qué palabras activan cada ruta, porque esa decisión define el comportamiento observable. Inspecciona `@mlflow.trace`. Verifica que cada pregunta devuelve texto y que en MLflow aparecen spans anidados.


In [0]:
@mlflow.trace
def route_question(question: str) -> str:
    text = question.lower()
    if any(word in text for word in ["token", "secreto", "contraseña", "privacidad"]):
        return "security"
    if any(word in text for word in ["producción", "api", "servicio"]):
        return "production"
    return "mlflow"


@mlflow.trace
def retrieve_course_context(topic: str) -> str:
    contexts = {
        "security": "No registres secretos en parámetros, tags ni artefactos.",
        "production": "En producción importan contratos, versionado y monitorización.",
        "mlflow": "Un experimento agrupa runs con parámetros, métricas y artefactos.",
    }
    return contexts[topic]


@mlflow.trace
def course_agent(question: str) -> str:
    route = route_question(question)
    context = retrieve_course_context(route)
    return f"Ruta: {route}. {context}"

questions = [
    "¿Qué objeto agrupa varios runs?",
    "¿Puedo pegar mi token en un parámetro de MLflow?",
    "¿Qué cambia cuando llevo un modelo a producción?",
]
answers = [{"question": q, "answer": course_agent(q)} for q in questions]
display(answers)
assert all("Ruta:" in item["answer"] for item in answers)

2026/10/04 10:17:15 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.


answer,question
"Ruta: mlflow. Un experimento agrupa runs con parámetros, métricas y artefactos.",¿Qué objeto agrupa varios runs?
"Ruta: security. No registres secretos en parámetros, tags ni artefactos.",¿Puedo pegar mi token en un parámetro de MLflow?
"Ruta: production. En producción importan contratos, versionado y monitorización.",¿Qué cambia cuando llevo un modelo a producción?


[Trace(trace_id=tr-4ab5ecca5bbd2418e0771fbbeed14d41), Trace(trace_id=tr-8821e5e356711eb020172bf9b010227e), Trace(trace_id=tr-be9268f6051848b5ebd08c62200f1e27)]

## 2. Inspeccionar una traza

En **Experiments**, abre una traza de `course_agent`. Anota qué ruta siguió la pregunta, qué contexto recuperó y qué cambiarías si una ruta fuese incorrecta.

![image_1791109808452.png](./image_1791109808452.png "image_1791109808452.png")

Evidentemente, se trata de un modelo muy sencillo que únicamente busca la palabra en la frase. Si un agente se equivocase de la ruta esperada, habría que depurar más que palabras son las que clasifican cada una de las rutas. Por otra parte, si es necesario, se podrían añadir más palabras necesarias para que acierte con la ruta


## 3. Evaluación mínima y transparente

No usamos un juez LLM. Comprobamos una expectativa simple: que la respuesta contenga una palabra esperada.

**TODO 2:** añade al menos dos casos: uno de privacidad/secretos y otro de producción. Importa porque la evaluación debe cubrir comportamientos relevantes. Inspecciona listas de diccionarios y verifica que `evaluation_df` tiene todos los casos.


In [0]:
EVALUATION_DATA = [
    {"question": "¿Qué objeto agrupa varios runs?", "must_include": "experimento"},
    {"question": "¿Puedo guardar una contraseña como tag?", "must_include": "secretos"},
    {"question": "¿Qué debo cuidar en producción?", "must_include": "contratos"},
    {"question": "¿Puedo subir un archivo de secretos .env a GitHub?", "must_include": "secretos"},
    {"question": "¿Se pueden desplegar archivos .ipynb en producción?", "must_include": "contratos"}
    # TODO 2: añade dos casos más y comprueba que siguen pasando.
]

evaluation_rows = []
for case in EVALUATION_DATA:
    answer = course_agent(case["question"])
    passed = case["must_include"].lower() in answer.lower()
    evaluation_rows.append({**case, "answer": answer, "passed": passed})

evaluation_df = __import__("pandas").DataFrame(evaluation_rows)
display(evaluation_df)
assert evaluation_df["passed"].all(), "Alguna expectativa mínima no se cumple"

question,must_include,answer,passed
¿Qué objeto agrupa varios runs?,experimento,"Ruta: mlflow. Un experimento agrupa runs con parámetros, métricas y artefactos.",true
¿Puedo guardar una contraseña como tag?,secretos,"Ruta: security. No registres secretos en parámetros, tags ni artefactos.",true
¿Qué debo cuidar en producción?,contratos,"Ruta: production. En producción importan contratos, versionado y monitorización.",true
¿Puedo subir un archivo de secretos .env a GitHub?,secretos,"Ruta: security. No registres secretos en parámetros, tags ni artefactos.",true
¿Se pueden desplegar archivos .ipynb en producción?,contratos,"Ruta: production. En producción importan contratos, versionado y monitorización.",true


[Trace(trace_id=tr-dce1c8db314826e038193eba73df9e74), Trace(trace_id=tr-8cbf730f92b85c4525721a445667e9e8), Trace(trace_id=tr-f407db7e40ede6e39adbdbd406bd8778), Trace(trace_id=tr-26b5716df67ac8ba6e42e8a73c0cb018), Trace(trace_id=tr-d74d117e7b269769fd141f261a63ed86)]

## 4. Registrar un run de evaluación

**TODO 3:** registra el número de casos y el porcentaje que pasa. Importa porque una evaluación sin run se pierde. Inspecciona `mlflow.log_metric` y `mlflow.log_dict`. Verifica que el run contiene el artefacto `evaluation/results.json`.


In [0]:
with mlflow.start_run(run_name=f"agent-evaluation-{AGENT_VERSION}") as run:
    mlflow.set_tags({"agent.version": AGENT_VERSION, "purpose": "guided-agentops-evaluation"})
    mlflow.log_param("case_count", len(evaluation_df))
    mlflow.log_metric("pass_rate", float(evaluation_df["passed"].mean()))
    mlflow.log_dict(evaluation_rows, "evaluation/results.json")
    EVALUATION_RUN_ID = run.info.run_id

print({"evaluation_run_id": EVALUATION_RUN_ID, "pass_rate": float(evaluation_df["passed"].mean())})

2026/10/04 10:19:51 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.


{'evaluation_run_id': 'ea6ea8cb2bf74d54970ec05cc9b60b35', 'pass_rate': 1.0}


## 5. Buscar trazas o runs por versión

**TODO 4:** filtra por `agent.version`. Importa porque un agente cambia rápido y necesitas comparar versiones. Inspecciona `mlflow.search_runs` y, si tu workspace lo permite, `mlflow.search_traces`. Verifica que sólo recuperas la versión `v1-deterministic`.


In [0]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"tags.agent.version = '{AGENT_VERSION}'",
    order_by=["attributes.start_time DESC"],
)
display(runs[["run_id", "tags.agent.version", "metrics.pass_rate"]])
assert set(runs["tags.agent.version"].dropna()) == {AGENT_VERSION}

run_id,tags.agent.version,metrics.pass_rate
ea6ea8cb2bf74d54970ec05cc9b60b35,v1-deterministic,1.0
d74082e4372f41f89636fc6db5c86a73,v1-deterministic,null
844f5de158974687bc5d246404c67adf,v1-deterministic,1.0
c32654cdfc904d66ba5b0ef1b939fd51,v1-deterministic,null
91c95778b04e4b65ae9386d0444f4d70,v1-deterministic,null


## Entregable

Incluye en la ficha: una captura de una traza, `EVALUATION_RUN_ID`, el `pass_rate` y una frase sobre qué riesgo cubrirías en la siguiente versión del agente.

evaluation_run_id = `ea6ea8cb2bf74d54970ec05cc9b60b35`

pass_rate = `1.0`

#### Ejecución de todas las trazas

![image_1791109320898.png](./image_1791109320898.png "image_1791109320898.png")

En una nueva versión, se podría añadir un contexto de buenas prácticas a la hora de programar. Por otra parte, habría que seguir mejorando el modelo para que incluyese más palabras y tuviese capacidad de distinguir mejor entre las diferentes preguntas.

